# Lecture 19: Unsupervised Learning

This notebook uses customer campaign features for PCA and clustering without using the response label during fitting.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")

from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data" / "raw"


In [ ]:
campaign = pd.read_csv(DATA / "campaign_response.csv")
features = ["age", "visits_last_month", "emails_opened", "discount_pct", "prior_spend_eur", "segment"]
numeric = ["age", "visits_last_month", "emails_opened", "discount_pct", "prior_spend_eur"]
categorical = ["segment"]


In [ ]:
preprocess = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ]
)
X = preprocess.fit_transform(campaign[features])


In [ ]:
pca = PCA(n_components=2, random_state=42)
components = pca.fit_transform(X)
pca_frame = pd.DataFrame(components, columns=["pc1", "pc2"])
pca_frame["responded"] = campaign["responded"]
print(pca.explained_variance_ratio_)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(data=pca_frame, x="pc1", y="pc2", hue="responded", alpha=0.75, ax=ax)
ax.set(title="PCA projection with response shown only after fitting")


In [ ]:
scores = []
for k in [2, 3, 4, 5, 6]:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)
    scores.append({"k": k, "silhouette": silhouette_score(X, labels)})
pd.DataFrame(scores)


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)
clustered = campaign.assign(cluster=labels)
clustered.groupby("cluster")[numeric + ["responded"]].mean().round(2)


## LLM Check

Ask an LLM to name each cluster using the summary table. Replace any unsupported label with a concrete description of the variables that differ.
